In [ ]:
try:
    import sieve
except ImportError:
    %pip install "sieve[chem] @ git+https://github.com/craabreu/sieve.git"
    import sieve

In [ ]:
from pathlib import Path
from dataclasses import replace

import numpy as np

from rdkit import Chem
from rdkit.Chem import Draw
from sieve.io.rdkit_adapter import from_rdkit

Select Sieve models

* `dash-qmugs`: Fitted to the higher-quality QMugs subset of the DASH dataset
* `dash-extra`: Fitted to the remaining, lower-quality subsets

**Note**: selecting both models means using the full DASH dataset

In [ ]:
selected_submodels = ["dash-qmugs", "dash-extra"]

Merge the parts of the selected submodels

In [ ]:
MODELS = Path("models")

model = sum(
    sieve.SieveModel.load(part)
    for submodel in selected_submodels
    for part in sorted((MODELS / submodel).glob("part-*.npz"))
)

In [ ]:
model = replace(model, config=replace(model.config, predictive_variance=True))

Define a batch of molecules

In [ ]:
SMILES = {
    "Aspirin": "CC(=O)Oc1ccccc1C(=O)O",
    "Paracetamol": "CC(=O)Nc1ccc(O)cc1",
    "Ibuprofen": "CC(C)Cc1ccc(C(C)C(=O)O)cc1",
    "Caffeine": "Cn1cnc2c1c(=O)n(C)c(=O)n2C",
    "Naproxen": "COc1ccc2cc(C(C)C(=O)O)ccc2c1",
}

In [ ]:
mols = [
    Chem.AddHs(Chem.MolFromSmiles(smi)) for smi in SMILES.values()
]

Draw.MolsToGridImage(
    [Chem.RemoveHs(m) for m in mols],
    legends=list(SMILES),
    molsPerRow=5,
    subImgSize=(200, 200),
)

Predict Partial Charges

In [ ]:
batch_prediction = model.predict_detailed(from_rdkit(mols, config=model.config))

In [ ]:
def split(array, mols):
    return [
        a.ravel()
        for a in np.split(
            array, np.cumsum([mol.GetNumAtoms() for mol in mols])[:-1]
        )
    ]

mol_charges = split(batch_prediction.value, mols)
mol_variances = split(batch_prediction.predictive_variance, mols)

Predicted partial charges sum up to the total formal charge approximately

In [ ]:
NORMALIZE = True

for mol, charges, variances in zip(mols, mol_charges, mol_variances):
    if NORMALIZE:
        total_charge = Chem.GetFormalCharge(mol)
        weights = variances / variances.sum()
        partial_charges = charges + weights * (total_charge - charges.sum())
    else:
        partial_charges = charges
    for i, atom in enumerate(mol.GetAtoms()):
        atom.SetProp("atomNote", f"{float(partial_charges[i]):+.2f}")

In [ ]:
opts = Draw.MolDrawOptions()
opts.annotationFontScale = 0.6

Draw.MolsToGridImage(
    mols,
    legends=list(SMILES),
    molsPerRow=2,
    subImgSize=(500, 300),
    drawOptions=opts,
)